# Automated Machine Learning for Classification Tasks

This notebook demonstrates automated machine learning workflows for classification problems using various algorithms and techniques. It's designed to work seamlessly in Google Colab.

## Features:
- Automated data preprocessing
- Multiple classification algorithms comparison
- Hyperparameter tuning
- Model evaluation and visualization
- Easy-to-use interface for different datasets

## 1. Setup and Installation
First, let's install required packages and import necessary libraries.

In [ ]:
# Install required packages for Google Colab
!pip install scikit-learn pandas numpy matplotlib seaborn plotly
!pip install xgboost lightgbm
!pip install imbalanced-learn

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn imports
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Classification algorithms
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb
import lightgbm as lgb

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve
)

# For handling imbalanced datasets
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("All libraries imported successfully!")

## 2. Data Loading and Exploration
Load your dataset and perform initial exploration.

In [ ]:
# Option 1: Load demo dataset (Iris dataset)
from sklearn.datasets import load_iris, load_wine, load_breast_cancer

# Choose one of the demo datasets
dataset_choice = 'breast_cancer'  # Options: 'iris', 'wine', 'breast_cancer'

if dataset_choice == 'iris':
    data = load_iris()
    df = pd.DataFrame(data.data, columns=data.feature_names)
    df['target'] = data.target
    target_names = data.target_names
elif dataset_choice == 'wine':
    data = load_wine()
    df = pd.DataFrame(data.data, columns=data.feature_names)
    df['target'] = data.target
    target_names = data.target_names
elif dataset_choice == 'breast_cancer':
    data = load_breast_cancer()
    df = pd.DataFrame(data.data, columns=data.feature_names)
    df['target'] = data.target
    target_names = data.target_names

print(f"Dataset: {dataset_choice}")
print(f"Shape: {df.shape}")
print(f"Target classes: {target_names}")
df.head()

In [ ]:
# Option 2: Upload your own CSV file
# Uncomment the following lines to upload your own dataset

# from google.colab import files
# uploaded = files.upload()
# filename = list(uploaded.keys())[0]
# df = pd.read_csv(filename)
# 
# # Specify your target column name
# target_column = 'target'  # Replace with your target column name
# 
# print(f"Dataset shape: {df.shape}")
# print("Columns:", df.columns.tolist())
# df.head()

In [ ]:
# Data exploration
print("Dataset Information:")
print(f"Shape: {df.shape}")
print(f"\nData types:")
print(df.dtypes)
print(f"\nMissing values:")
print(df.isnull().sum().sum())
print(f"\nTarget distribution:")
print(df['target'].value_counts())

# Basic statistics
df.describe()

In [ ]:
# Visualize target distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Target distribution
df['target'].value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Target Distribution')
axes[0].set_xlabel('Target Class')
axes[0].set_ylabel('Count')

# Target distribution pie chart
df['target'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
axes[1].set_title('Target Distribution (Percentage)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

# Correlation heatmap for numerical features
numerical_cols = df.select_dtypes(include=[np.number]).columns
if len(numerical_cols) > 1:
    plt.figure(figsize=(12, 8))
    correlation_matrix = df[numerical_cols].corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
    plt.title('Feature Correlation Heatmap')
    plt.show()

## 3. Automated Data Preprocessing
Prepare the data for machine learning with automated preprocessing steps.

In [ ]:
class AutomatedPreprocessor:
    def __init__(self):
        self.preprocessor = None
        self.label_encoder = None
        
    def fit_transform(self, X, y):
        # Separate numerical and categorical features
        numerical_features = X.select_dtypes(include=[np.number]).columns.tolist()
        categorical_features = X.select_dtypes(include=['object']).columns.tolist()
        
        print(f"Numerical features: {numerical_features}")
        print(f"Categorical features: {categorical_features}")
        
        # Create preprocessing pipelines
        numerical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])
        
        categorical_transformer = Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ])
        
        # Combine preprocessors
        self.preprocessor = ColumnTransformer(
            transformers=[
                ('num', numerical_transformer, numerical_features),
                ('cat', categorical_transformer, categorical_features)
            ]
        )
        
        # Fit and transform features
        X_processed = self.preprocessor.fit_transform(X)
        
        # Encode target if it's categorical
        if y.dtype == 'object':
            self.label_encoder = LabelEncoder()
            y_processed = self.label_encoder.fit_transform(y)
        else:
            y_processed = y
            
        return X_processed, y_processed
    
    def transform(self, X, y=None):
        X_processed = self.preprocessor.transform(X)
        
        if y is not None and self.label_encoder is not None:
            y_processed = self.label_encoder.transform(y)
            return X_processed, y_processed
        return X_processed

# Prepare features and target
X = df.drop('target', axis=1)
y = df['target']

# Apply preprocessing
preprocessor = AutomatedPreprocessor()
X_processed, y_processed = preprocessor.fit_transform(X, y)

print(f"\nOriginal shape: {X.shape}")
print(f"Processed shape: {X_processed.shape}")
print(f"Target classes: {np.unique(y_processed)}")

In [ ]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y_processed, test_size=0.2, random_state=42, stratify=y_processed
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print(f"Training target distribution: {np.bincount(y_train)}")
print(f"Test target distribution: {np.bincount(y_test)}")

## 4. Automated Model Training and Comparison
Train multiple classification models and compare their performance.

In [ ]:
# Define classification models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'SVM': SVC(random_state=42, probability=True),
    'K-Nearest Neighbors': KNeighborsClassifier(),
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'XGBoost': xgb.XGBClassifier(random_state=42, eval_metric='logloss'),
    'LightGBM': lgb.LGBMClassifier(random_state=42, verbose=-1)
}

print(f"Total models to train: {len(models)}")

In [ ]:
# Train and evaluate all models
results = {}
cv_scores = {}

# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Training models...\n")

for name, model in models.items():
    print(f"Training {name}...")
    
    # Cross-validation
    cv_score = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    cv_scores[name] = cv_score
    
    # Train on full training set
    model.fit(X_train, y_train)
    
    # Predictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1] if len(np.unique(y_processed)) == 2 else None
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    # AUC for binary classification
    if len(np.unique(y_processed)) == 2 and y_pred_proba is not None:
        auc = roc_auc_score(y_test, y_pred_proba)
    else:
        auc = None
    
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'auc': auc,
        'cv_mean': cv_score.mean(),
        'cv_std': cv_score.std(),
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }
    
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  CV Score: {cv_score.mean():.4f} (+/- {cv_score.std() * 2:.4f})")
    print()

print("All models trained successfully!")

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[model]['accuracy'] for model in results.keys()],
    'Precision': [results[model]['precision'] for model in results.keys()],
    'Recall': [results[model]['recall'] for model in results.keys()],
    'F1-Score': [results[model]['f1_score'] for model in results.keys()],
    'AUC': [results[model]['auc'] for model in results.keys()],
    'CV Mean': [results[model]['cv_mean'] for model in results.keys()],
    'CV Std': [results[model]['cv_std'] for model in results.keys()]
})

# Sort by accuracy
results_df = results_df.sort_values('Accuracy', ascending=False)
print("Model Performance Comparison:")
print(results_df.round(4))

# Find best model
best_model_name = results_df.iloc[0]['Model']
best_model = results[best_model_name]['model']
print(f"\nBest Model: {best_model_name}")
print(f"Best Accuracy: {results_df.iloc[0]['Accuracy']:.4f}")

## 5. Model Performance Visualization

In [ ]:
# Performance comparison plots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Accuracy comparison
results_df.plot(x='Model', y='Accuracy', kind='bar', ax=axes[0,0], color='skyblue')
axes[0,0].set_title('Model Accuracy Comparison')
axes[0,0].set_ylabel('Accuracy')
axes[0,0].tick_params(axis='x', rotation=45)

# F1-Score comparison
results_df.plot(x='Model', y='F1-Score', kind='bar', ax=axes[0,1], color='lightgreen')
axes[0,1].set_title('Model F1-Score Comparison')
axes[0,1].set_ylabel('F1-Score')
axes[0,1].tick_params(axis='x', rotation=45)

# Cross-validation scores
results_df.plot(x='Model', y='CV Mean', kind='bar', ax=axes[1,0], color='orange')
axes[1,0].set_title('Cross-Validation Accuracy')
axes[1,0].set_ylabel('CV Accuracy')
axes[1,0].tick_params(axis='x', rotation=45)

# AUC comparison (if binary classification)
if results_df['AUC'].notna().any():
    results_df_auc = results_df.dropna(subset=['AUC'])
    results_df_auc.plot(x='Model', y='AUC', kind='bar', ax=axes[1,1], color='pink')
    axes[1,1].set_title('Model AUC Comparison')
    axes[1,1].set_ylabel('AUC')
    axes[1,1].tick_params(axis='x', rotation=45)
else:
    axes[1,1].text(0.5, 0.5, 'AUC not available\n(Multi-class problem)', 
                   ha='center', va='center', transform=axes[1,1].transAxes)
    axes[1,1].set_title('AUC Not Available')

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix for best model
best_predictions = results[best_model_name]['predictions']
cm = confusion_matrix(y_test, best_predictions)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names if 'target_names' in locals() else range(len(np.unique(y_test))),
            yticklabels=target_names if 'target_names' in locals() else range(len(np.unique(y_test))))
plt.title(f'Confusion Matrix - {best_model_name}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

# Classification report
print(f"Classification Report - {best_model_name}:")
print(classification_report(y_test, best_predictions, 
                          target_names=target_names if 'target_names' in locals() else None))

In [ ]:
# ROC Curve for binary classification
if len(np.unique(y_processed)) == 2:
    plt.figure(figsize=(12, 5))
    
    # ROC Curve
    plt.subplot(1, 2, 1)
    for name in ['Logistic Regression', 'Random Forest', 'SVM', best_model_name]:
        if name in results and results[name]['probabilities'] is not None:
            fpr, tpr, _ = roc_curve(y_test, results[name]['probabilities'])
            auc_score = results[name]['auc']
            plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_score:.3f})')
    
    plt.plot([0, 1], [0, 1], 'k--', label='Random')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves')
    plt.legend()
    plt.grid(True)
    
    # Precision-Recall Curve
    plt.subplot(1, 2, 2)
    for name in ['Logistic Regression', 'Random Forest', 'SVM', best_model_name]:
        if name in results and results[name]['probabilities'] is not None:
            precision, recall, _ = precision_recall_curve(y_test, results[name]['probabilities'])
            plt.plot(recall, precision, label=name)
    
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curves')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.show()
else:
    print("ROC curves are only available for binary classification problems.")

## 6. Hyperparameter Tuning for Best Model
Optimize the best performing model using grid search.

In [ ]:
# Define hyperparameter grids for different models
param_grids = {
    'Random Forest': {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5, 10]
    },
    'Logistic Regression': {
        'C': [0.1, 1, 10, 100],
        'solver': ['liblinear', 'lbfgs']
    },
    'SVM': {
        'C': [0.1, 1, 10],
        'kernel': ['rbf', 'linear'],
        'gamma': ['scale', 'auto']
    },
    'XGBoost': {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 6, 9],
        'learning_rate': [0.01, 0.1, 0.2]
    }
}

# Perform hyperparameter tuning for the best model
if best_model_name in param_grids:
    print(f"Performing hyperparameter tuning for {best_model_name}...")
    
    # Get the model class
    base_model = type(best_model)()
    if hasattr(base_model, 'random_state'):
        base_model.set_params(random_state=42)
    
    # Grid search
    grid_search = GridSearchCV(
        base_model,
        param_grids[best_model_name],
        cv=5,
        scoring='accuracy',
        n_jobs=-1,
        verbose=1
    )
    
    grid_search.fit(X_train, y_train)
    
    # Best parameters and score
    print(f"\nBest parameters: {grid_search.best_params_}")
    print(f"Best cross-validation score: {grid_search.best_score_:.4f}")
    
    # Evaluate tuned model
    tuned_model = grid_search.best_estimator_
    tuned_predictions = tuned_model.predict(X_test)
    tuned_accuracy = accuracy_score(y_test, tuned_predictions)
    
    print(f"\nOriginal {best_model_name} accuracy: {results[best_model_name]['accuracy']:.4f}")
    print(f"Tuned {best_model_name} accuracy: {tuned_accuracy:.4f}")
    print(f"Improvement: {tuned_accuracy - results[best_model_name]['accuracy']:.4f}")
    
    # Update best model if improved
    if tuned_accuracy > results[best_model_name]['accuracy']:
        best_model = tuned_model
        print("\nUpdated best model with tuned parameters!")
    
else:
    print(f"Hyperparameter tuning not implemented for {best_model_name}")
    print("You can add custom parameter grids for this model.")

## 7. Feature Importance Analysis

In [ ]:
# Feature importance for tree-based models
if hasattr(best_model, 'feature_importances_'):
    # Get feature names
    feature_names = []
    
    # Numerical features
    numerical_features = X.select_dtypes(include=[np.number]).columns.tolist()
    feature_names.extend(numerical_features)
    
    # Categorical features (if any)
    categorical_features = X.select_dtypes(include=['object']).columns.tolist()
    if categorical_features:
        # Get feature names from preprocessor
        if hasattr(preprocessor.preprocessor.named_transformers_['cat'], 'named_steps'):
            encoder = preprocessor.preprocessor.named_transformers_['cat'].named_steps['onehot']
            if hasattr(encoder, 'get_feature_names_out'):
                cat_feature_names = encoder.get_feature_names_out(categorical_features)
                feature_names.extend(cat_feature_names)
    
    # If we couldn't get proper feature names, use generic ones
    if len(feature_names) != X_processed.shape[1]:
        feature_names = [f'Feature_{i}' for i in range(X_processed.shape[1])]
    
    # Create feature importance DataFrame
    feature_importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': best_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    # Plot feature importance
    plt.figure(figsize=(10, 8))
    top_features = feature_importance_df.head(15)  # Top 15 features
    sns.barplot(data=top_features, y='feature', x='importance')
    plt.title(f'Top Feature Importances - {best_model_name}')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()
    
    print("Top 10 Most Important Features:")
    print(feature_importance_df.head(10))
    
elif hasattr(best_model, 'coef_'):
    # For linear models, show coefficients
    print(f"Model coefficients for {best_model_name}:")
    coefficients = best_model.coef_
    if len(coefficients.shape) > 1:
        coefficients = coefficients[0]  # For binary classification
    
    # Create feature names (simplified)
    feature_names = [f'Feature_{i}' for i in range(len(coefficients))]
    
    coef_df = pd.DataFrame({
        'feature': feature_names,
        'coefficient': coefficients
    }).sort_values('coefficient', key=abs, ascending=False)
    
    print(coef_df.head(10))
    
else:
    print(f"Feature importance not available for {best_model_name}")

## 8. Handling Imbalanced Datasets (Optional)
If your dataset is imbalanced, try these techniques.

In [ ]:
# Check if dataset is imbalanced
class_counts = np.bincount(y_train)
imbalance_ratio = class_counts.max() / class_counts.min()

print(f"Class distribution: {class_counts}")
print(f"Imbalance ratio: {imbalance_ratio:.2f}")

if imbalance_ratio > 2:
    print("\nDataset appears to be imbalanced. Applying resampling techniques...")
    
    # SMOTE oversampling
    smote = SMOTE(random_state=42)
    X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
    
    # Train best model on balanced data
    balanced_model = type(best_model)()
    if hasattr(balanced_model, 'random_state'):
        balanced_model.set_params(random_state=42)
    
    balanced_model.fit(X_train_smote, y_train_smote)
    balanced_predictions = balanced_model.predict(X_test)
    balanced_accuracy = accuracy_score(y_test, balanced_predictions)
    
    print(f"\nOriginal model accuracy: {results[best_model_name]['accuracy']:.4f}")
    print(f"Balanced model accuracy: {balanced_accuracy:.4f}")
    
    # Compare classification reports
    print("\nOriginal Model Classification Report:")
    print(classification_report(y_test, results[best_model_name]['predictions']))
    
    print("\nBalanced Model Classification Report:")
    print(classification_report(y_test, balanced_predictions))
    
else:
    print("\nDataset is relatively balanced. No resampling needed.")

## 9. Model Saving and Loading
Save the best model for future use.

In [ ]:
import pickle
import joblib

# Save the best model and preprocessor
model_filename = f'best_classification_model_{best_model_name.lower().replace(" ", "_")}.pkl'
preprocessor_filename = 'classification_preprocessor.pkl'

# Save using joblib (recommended for scikit-learn models)
joblib.dump(best_model, model_filename)
joblib.dump(preprocessor, preprocessor_filename)

print(f"Model saved as: {model_filename}")
print(f"Preprocessor saved as: {preprocessor_filename}")

# Demonstrate loading
loaded_model = joblib.load(model_filename)
loaded_preprocessor = joblib.load(preprocessor_filename)

# Test loaded model
test_predictions = loaded_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_predictions)

print(f"\nLoaded model accuracy: {test_accuracy:.4f}")
print("Model loading successful!")

## 10. Create Prediction Function
Create a simple function to make predictions on new data.

In [ ]:
def predict_new_data(new_data, model_path=None, preprocessor_path=None):
    """
    Make predictions on new data using the trained model.
    
    Parameters:
    new_data: pandas DataFrame with the same features as training data
    model_path: path to saved model (optional)
    preprocessor_path: path to saved preprocessor (optional)
    
    Returns:
    predictions: array of predicted classes
    probabilities: array of prediction probabilities (if available)
    """
    
    # Load model and preprocessor if paths provided
    if model_path and preprocessor_path:
        model = joblib.load(model_path)
        prep = joblib.load(preprocessor_path)
    else:
        # Use the current best model and preprocessor
        model = best_model
        prep = preprocessor
    
    # Preprocess the new data
    new_data_processed = prep.transform(new_data)
    
    # Make predictions
    predictions = model.predict(new_data_processed)
    
    # Get probabilities if available
    try:
        probabilities = model.predict_proba(new_data_processed)
    except:
        probabilities = None
    
    # Convert predictions back to original labels if needed
    if hasattr(prep, 'label_encoder') and prep.label_encoder is not None:
        predictions = prep.label_encoder.inverse_transform(predictions)
    
    return predictions, probabilities

# Example usage with a sample from the test set
sample_data = X.iloc[:3]  # Take first 3 samples
sample_predictions, sample_probabilities = predict_new_data(sample_data)

print("Example predictions on sample data:")
print(f"Predictions: {sample_predictions}")
if sample_probabilities is not None:
    print(f"Probabilities shape: {sample_probabilities.shape}")
    print(f"First sample probabilities: {sample_probabilities[0]}")

print("\nPrediction function created successfully!")

## 11. Summary and Next Steps

### What we accomplished:
1. ✅ Automated data preprocessing pipeline
2. ✅ Trained and compared 10 different classification models
3. ✅ Performed hyperparameter tuning on the best model
4. ✅ Visualized model performance and feature importance
5. ✅ Handled potential class imbalance
6. ✅ Created model saving/loading functionality
7. ✅ Built a prediction function for new data

### Key Results:

In [ ]:
# Final summary
print("=" * 60)
print("AUTOMATED CLASSIFICATION ML - FINAL SUMMARY")
print("=" * 60)
print(f"Dataset: {dataset_choice if 'dataset_choice' in locals() else 'Custom dataset'}")
print(f"Number of samples: {df.shape[0]}")
print(f"Number of features: {df.shape[1] - 1}")
print(f"Number of classes: {len(np.unique(y_processed))}")
print()
print(f"Best Model: {best_model_name}")
print(f"Best Accuracy: {results[best_model_name]['accuracy']:.4f}")
print(f"Best F1-Score: {results[best_model_name]['f1_score']:.4f}")
if results[best_model_name]['auc'] is not None:
    print(f"Best AUC: {results[best_model_name]['auc']:.4f}")
print()
print("Top 3 Models:")
for i, row in results_df.head(3).iterrows():
    print(f"  {i+1}. {row['Model']}: {row['Accuracy']:.4f}")
print()
print("Files saved:")
print(f"  - {model_filename}")
print(f"  - {preprocessor_filename}")
print("=" * 60)

# Provide Google Colab specific tips
print("\n📝 GOOGLE COLAB TIPS:")
print("1. To use your own dataset, upload a CSV file using the file upload cell")
print("2. Save important files to Google Drive to persist them")
print("3. Use GPU runtime for faster training on large datasets")
print("4. Install additional packages as needed for your specific use case")
print("5. Consider using AutoML libraries like H2O or Auto-sklearn for even more automation")